# 🔍 Validación de Estructura Actual - Post Revert

Este notebook valida que la aplicación funcione correctamente con la estructura de modelos restaurada (catalog/ + main/) después del revert de la consolidación.

## 📋 Objetivos:
1. **Validar estructura de modelos** - Verificar que catalog/ y main/ estén correctos
2. **Revisar servicios** - Asegurar que usen rutas de importación correctas  
3. **Probar funcionalidad** - Verificar que la aplicación funcione completamente
4. **Preparar consolidación** - Identificar mejores prácticas para futura consolidación

## 🎯 Estado Actual:
- **Commit:** `17753b5` (antes de consolidación)
- **Estructura:** `src/models/catalog/` + `src/models/main/`
- **Objetivo:** Validación completa antes de nueva consolidación

## 1. 📚 Import Required Libraries

Importamos las librerías necesarias para testing, requests, manejo de configuración y logging.

In [ ]:
// Importar librerías necesarias para validación
const fs = require('fs');
const path = require('path');
const axios = require('axios');
require('dotenv').config();

// Variables globales para tracking
let validacion_resultado = {
    estructura_modelos: {},
    servicios_rutas: {},
    endpoints_api: {},
    dependencias: {},
    integracion: {},
    errores: [],
    warnings: []
};

console.log('✅ Librerías importadas correctamente');
console.log('📋 Sistema de validación inicializado');

// Función helper para logging con colores
function log_step(step, mensaje, tipo = 'info') {
    const timestamp = new Date().toISOString();
    const icon = tipo === 'success' ? '✅' : tipo === 'error' ? '❌' : tipo === 'warning' ? '⚠️' : 'ℹ️';
    console.log(`${icon} [${step}] ${mensaje}`);
    
    if (tipo === 'error') {
        validacion_resultado.errores.push(`[${step}] ${mensaje}`);
    } else if (tipo === 'warning') {
        validacion_resultado.warnings.push(`[${step}] ${mensaje}`);
    }
}

## 2. ⚙️ Load Application Configuration

Cargamos y validamos los archivos de configuración y variables de entorno actuales.

In [ ]:
// Cargar y validar configuración de la aplicación
log_step('2.1', 'Cargando configuración de la aplicación...');

// Verificar archivos de configuración esenciales
const archivos_config = [
    'package.json',
    '.env',
    'src/app.js',
    'src/config/sequelize.js',
    'src/models/index.js'
];

validacion_resultado.estructura_modelos.archivos_config = {};

for (const archivo of archivos_config) {
    try {
        if (fs.existsSync(archivo)) {
            const stats = fs.statSync(archivo);
            validacion_resultado.estructura_modelos.archivos_config[archivo] = {
                existe: true,
                tamaño: stats.size,
                modificado: stats.mtime
            };
            log_step('2.1', `✅ ${archivo} encontrado (${stats.size} bytes)`);
        } else {
            validacion_resultado.estructura_modelos.archivos_config[archivo] = {
                existe: false
            };
            log_step('2.1', `❌ ${archivo} no encontrado`, 'error');
        }
    } catch (error) {
        log_step('2.1', `❌ Error leyendo ${archivo}: ${error.message}`, 'error');
    }
}

// Verificar variables de entorno críticas
log_step('2.2', 'Verificando variables de entorno...');

const env_vars_criticas = ['DB_HOST', 'DB_USER', 'DB_PASSWORD', 'DB_NAME', 'JWT_SECRET'];
validacion_resultado.estructura_modelos.env_vars = {};

for (const variable of env_vars_criticas) {
    if (process.env[variable]) {
        validacion_resultado.estructura_modelos.env_vars[variable] = 'configurada';
        log_step('2.2', `✅ ${variable} configurada`);
    } else {
        validacion_resultado.estructura_modelos.env_vars[variable] = 'faltante';
        log_step('2.2', `❌ ${variable} no configurada`, 'error');
    }
}

log_step('2.3', 'Configuración cargada y validada');

## 3. 🏗️ Validar Estructura de Modelos

Verificamos que la estructura actual de modelos (catalog/ + main/) esté correcta y completa.

In [ ]:
// Validar estructura completa de modelos
log_step('3.1', 'Analizando estructura de modelos...');

// Verificar directorios principales
const directorios_modelos = [
    'src/models',
    'src/models/catalog',
    'src/models/main'
];

validacion_resultado.estructura_modelos.directorios = {};

for (const directorio of directorios_modelos) {
    if (fs.existsSync(directorio)) {
        const archivos = fs.readdirSync(directorio);
        validacion_resultado.estructura_modelos.directorios[directorio] = {
            existe: true,
            archivos: archivos,
            total: archivos.length
        };
        log_step('3.1', `✅ ${directorio} encontrado con ${archivos.length} archivos`);
    } else {
        validacion_resultado.estructura_modelos.directorios[directorio] = {
            existe: false
        };
        log_step('3.1', `❌ ${directorio} no encontrado`, 'error');
    }
}

// Analizar modelos en catalog/ (ES6 modules)
log_step('3.2', 'Analizando modelos catalog/ (ES6)...');

if (validacion_resultado.estructura_modelos.directorios['src/models/catalog']) {
    const modelos_catalog = validacion_resultado.estructura_modelos.directorios['src/models/catalog'].archivos
        .filter(archivo => archivo.endsWith('.js'));
    
    validacion_resultado.estructura_modelos.catalog_models = {
        total: modelos_catalog.length,
        archivos: modelos_catalog,
        tipos: {}
    };
    
    // Verificar algunos modelos clave
    const modelos_clave_catalog = ['Familias.js', 'Persona.js', 'Departamentos.js', 'Municipios.js'];
    for (const modelo of modelos_clave_catalog) {
        if (modelos_catalog.includes(modelo)) {
            log_step('3.2', `✅ Modelo clave encontrado: ${modelo}`);
            validacion_resultado.estructura_modelos.catalog_models.tipos[modelo] = 'encontrado';
        } else {
            log_step('3.2', `❌ Modelo clave faltante: ${modelo}`, 'error');
            validacion_resultado.estructura_modelos.catalog_models.tipos[modelo] = 'faltante';
        }
    }
}

// Analizar modelos en main/ (CommonJS)
log_step('3.3', 'Analizando modelos main/ (CommonJS)...');

if (validacion_resultado.estructura_modelos.directorios['src/models/main']) {
    const modelos_main = validacion_resultado.estructura_modelos.directorios['src/models/main'].archivos
        .filter(archivo => archivo.endsWith('.cjs'));
    
    validacion_resultado.estructura_modelos.main_models = {
        total: modelos_main.length,
        archivos: modelos_main,
        tipos: {}
    };
    
    // Verificar algunos modelos clave
    const modelos_clave_main = ['Familia.cjs', 'Persona.cjs', 'Departamento.cjs', 'Encuesta.cjs'];
    for (const modelo of modelos_clave_main) {
        if (modelos_main.includes(modelo)) {
            log_step('3.3', `✅ Modelo clave encontrado: ${modelo}`);
            validacion_resultado.estructura_modelos.main_models.tipos[modelo] = 'encontrado';
        } else {
            log_step('3.3', `❌ Modelo clave faltante: ${modelo}`, 'warning');
            validacion_resultado.estructura_modelos.main_models.tipos[modelo] = 'faltante';
        }
    }
}

// Verificar modelos base
log_step('3.4', 'Verificando modelos base...');

const modelos_base = ['Role.js', 'Usuario.js', 'UsuarioRole.js'];
validacion_resultado.estructura_modelos.base_models = {};

for (const modelo of modelos_base) {
    const ruta = `src/models/${modelo}`;
    if (fs.existsSync(ruta)) {
        validacion_resultado.estructura_modelos.base_models[modelo] = 'encontrado';
        log_step('3.4', `✅ Modelo base: ${modelo}`);
    } else {
        validacion_resultado.estructura_modelos.base_models[modelo] = 'faltante';
        log_step('3.4', `❌ Modelo base faltante: ${modelo}`, 'error');
    }
}

log_step('3.5', 'Análisis de estructura de modelos completado');

## 4. 🗄️ Test Database Connection

Verificamos la conectividad con la base de datos y probamos operaciones CRUD básicas.

In [ ]:
// Test de conexión a base de datos
log_step('4.1', 'Probando conexión a base de datos...');

async function testDatabaseConnection() {
    try {
        // Importar Sequelize
        const { Sequelize } = require('sequelize');
        
        // Crear conexión
        const sequelize = new Sequelize(
            process.env.DB_NAME,
            process.env.DB_USER,
            process.env.DB_PASSWORD,
            {
                host: process.env.DB_HOST,
                port: process.env.DB_PORT || 5432,
                dialect: 'postgres',
                logging: false
            }
        );
        
        // Test de autenticación
        await sequelize.authenticate();
        log_step('4.1', '✅ Conexión a base de datos exitosa');
        
        // Test de consulta básica
        const [results] = await sequelize.query('SELECT version()');
        log_step('4.2', `✅ PostgreSQL Version: ${results[0].version.split(' ')[0]} ${results[0].version.split(' ')[1]}`);
        
        // Verificar tablas existentes
        const [tables] = await sequelize.query(`
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public' 
            AND table_type = 'BASE TABLE'
            ORDER BY table_name
        `);
        
        validacion_resultado.dependencias.database = {
            conectado: true,
            version: results[0].version,
            tablas_total: tables.length,
            tablas: tables.map(t => t.table_name)
        };
        
        log_step('4.3', `✅ Base de datos contiene ${tables.length} tablas`);
        
        // Verificar tablas clave
        const tablas_clave = ['usuarios', 'roles', 'familias', 'personas', 'departamentos'];
        const tablas_encontradas = tables.map(t => t.table_name.toLowerCase());
        
        for (const tabla of tablas_clave) {
            if (tablas_encontradas.includes(tabla)) {
                log_step('4.3', `✅ Tabla clave encontrada: ${tabla}`);
            } else {
                log_step('4.3', `⚠️ Tabla clave no encontrada: ${tabla}`, 'warning');
            }
        }
        
        await sequelize.close();
        return true;
        
    } catch (error) {
        log_step('4.1', `❌ Error de conexión: ${error.message}`, 'error');
        validacion_resultado.dependencias.database = {
            conectado: false,
            error: error.message
        };
        return false;
    }
}

// Ejecutar test de base de datos
const db_conectada = await testDatabaseConnection();

## 5. 🛣️ Validate Service Routes

Verificamos todas las rutas de servicios y endpoints para asegurar que estén correctamente configurados.

In [ ]:
// Validar rutas de servicios y imports
log_step('5.1', 'Analizando servicios y sus importaciones...');

function analizarImportsServicio(rutaArchivo) {
    try {
        const contenido = fs.readFileSync(rutaArchivo, 'utf8');
        const imports = [];
        
        // Buscar imports de modelos
        const importRegex = /(?:import|require)\s*\(?[^)]*\)?\s*from\s*['"](.*?)['"]|require\s*\(\s*['"](.*?)['"]\s*\)/g;
        let match;
        
        while ((match = importRegex.exec(contenido)) !== null) {
            const importPath = match[1] || match[2];
            if (importPath && (importPath.includes('../models') || importPath.includes('./models'))) {
                imports.push(importPath);
            }
        }
        
        return {
            archivo: rutaArchivo,
            imports_modelos: imports,
            total_imports: imports.length,
            contenido_size: contenido.length
        };
    } catch (error) {
        return {
            archivo: rutaArchivo,
            error: error.message
        };
    }
}

// Encontrar todos los servicios
const directorios_servicios = [
    'src/services',
    'src/services/catalog',
    'src/controllers'
];

validacion_resultado.servicios_rutas = {
    total_servicios: 0,
    servicios_analizados: [],
    imports_problematicos: [],
    rutas_consolidadas: 0
};

for (const directorio of directorios_servicios) {
    if (fs.existsSync(directorio)) {
        const archivos = fs.readdirSync(directorio)
            .filter(archivo => archivo.endsWith('.js'))
            .map(archivo => path.join(directorio, archivo));
        
        log_step('5.1', `📁 Analizando ${archivos.length} servicios en ${directorio}`);
        
        for (const archivo of archivos) {
            const analisis = analizarImportsServicio(archivo);
            validacion_resultado.servicios_rutas.servicios_analizados.push(analisis);
            validacion_resultado.servicios_rutas.total_servicios++;
            
            if (analisis.imports_modelos) {
                // Verificar si usa rutas de consolidated (problemático)
                const imports_consolidated = analisis.imports_modelos.filter(imp => 
                    imp.includes('consolidated'));
                
                if (imports_consolidated.length > 0) {
                    validacion_resultado.servicios_rutas.imports_problematicos.push({
                        archivo: archivo,
                        imports_consolidated: imports_consolidated
                    });
                    validacion_resultado.servicios_rutas.rutas_consolidadas += imports_consolidated.length;
                    log_step('5.1', `⚠️ ${path.basename(archivo)}: ${imports_consolidated.length} imports a 'consolidated'`, 'warning');
                } else {
                    log_step('5.1', `✅ ${path.basename(archivo)}: ${analisis.imports_modelos.length} imports correctos`);
                }
            }
        }
    } else {
        log_step('5.1', `❌ Directorio no encontrado: ${directorio}`, 'error');
    }
}

log_step('5.2', `Análisis completado: ${validacion_resultado.servicios_rutas.total_servicios} servicios`);

if (validacion_resultado.servicios_rutas.rutas_consolidadas > 0) {
    log_step('5.2', `⚠️ ${validacion_resultado.servicios_rutas.rutas_consolidadas} imports problemáticos encontrados`, 'warning');
} else {
    log_step('5.2', '✅ Todos los imports de servicios son correctos');
}

## 6. 📦 Check Library Dependencies
Verificar que todas las dependencias del proyecto estén correctamente instaladas y compatibles.

In [ ]:
// Verificar dependencias del proyecto
log_step('6.1', 'Verificando package.json y dependencias...');

validacion_resultado.dependencias = {
    package_json_exists: false,
    node_modules_exists: false,
    dependencias_principales: [],
    dependencias_faltantes: [],
    version_node: null,
    compatible: true
};

// Verificar package.json
if (fs.existsSync('package.json')) {
    validacion_resultado.dependencias.package_json_exists = true;
    log_step('6.1', '✅ package.json encontrado');
    
    try {
        const packageJson = JSON.parse(fs.readFileSync('package.json', 'utf8'));
        
        // Obtener versión de Node.js
        validacion_resultado.dependencias.version_node = process.version;
        log_step('6.1', `Node.js version: ${process.version}`);
        
        // Verificar dependencias principales
        const dependencias_criticas = [
            'express', 'sequelize', 'pg', 'pg-hstore',
            'bcryptjs', 'jsonwebtoken', 'cors', 'helmet',
            'express-validator', 'multer', 'nodemailer'
        ];
        
        const dependencies = { ...packageJson.dependencies, ...packageJson.devDependencies };
        
        for (const dep of dependencias_criticas) {
            if (dependencies[dep]) {
                validacion_resultado.dependencias.dependencias_principales.push({
                    nombre: dep,
                    version: dependencies[dep],
                    instalado: fs.existsSync(path.join('node_modules', dep))
                });
                log_step('6.1', `📦 ${dep}: ${dependencies[dep]}`);
            } else {
                validacion_resultado.dependencias.dependencias_faltantes.push(dep);
                log_step('6.1', `❌ Dependencia faltante: ${dep}`, 'error');
                validacion_resultado.dependencias.compatible = false;
            }
        }
        
    } catch (error) {
        log_step('6.1', `❌ Error leyendo package.json: ${error.message}`, 'error');
        validacion_resultado.dependencias.compatible = false;
    }
} else {
    log_step('6.1', '❌ package.json no encontrado', 'error');
    validacion_resultado.dependencias.compatible = false;
}

// Verificar node_modules
if (fs.existsSync('node_modules')) {
    validacion_resultado.dependencias.node_modules_exists = true;
    log_step('6.2', '✅ node_modules encontrado');
    
    // Contar paquetes instalados
    const paquetes = fs.readdirSync('node_modules').filter(dir => 
        !dir.startsWith('.') && fs.statSync(path.join('node_modules', dir)).isDirectory()
    );
    log_step('6.2', `📦 ${paquetes.length} paquetes instalados`);
} else {
    log_step('6.2', '❌ node_modules no encontrado - ejecutar npm install', 'error');
    validacion_resultado.dependencias.compatible = false;
}

// Verificar compatibilidad de versiones
if (validacion_resultado.dependencias.version_node) {
    const nodeVersion = parseInt(validacion_resultado.dependencias.version_node.substring(1));
    if (nodeVersion < 14) {
        log_step('6.3', `⚠️ Node.js ${nodeVersion} puede ser incompatible (recomendado >= 14)`, 'warning');
        validacion_resultado.dependencias.compatible = false;
    } else {
        log_step('6.3', `✅ Node.js ${nodeVersion} es compatible`);
    }
}

log_step('6.4', validacion_resultado.dependencias.compatible ? 
    '✅ Todas las dependencias están correctas' : 
    '❌ Hay problemas con las dependencias');

console.log('📋 Resumen de dependencias:', {
    total_principales: validacion_resultado.dependencias.dependencias_principales.length,
    faltantes: validacion_resultado.dependencias.dependencias_faltantes.length,
    compatible: validacion_resultado.dependencias.compatible
});

## 7. 🔬 Integration Tests
Ejecutar pruebas básicas de integración para verificar el funcionamiento del sistema.

In [ ]:
// Pruebas de integración básicas
log_step('7.1', 'Iniciando pruebas de integración...');

validacion_resultado.integracion = {
    modelos_cargables: 0,
    servicios_funcionales: 0,
    rutas_disponibles: 0,
    errores_criticos: [],
    warnings: [],
    pruebas_exitosas: []
};

// Prueba 1: Cargar modelos principales
log_step('7.1', 'Probando carga de modelos principales...');

const modelos_criticos = [
    'Usuario', 'Role', 'UsuarioRole', 'Departamento', 
    'Municipio', 'Parroquia', 'Sector'
];

for (const modelo of modelos_criticos) {
    try {
        // Intentar cargar desde catalog
        let modeloCargado = null;
        const rutaCatalog = `./src/models/catalog/${modelo}.js`;
        const rutaMain = `./src/models/main/${modelo}.cjs`;
        
        if (fs.existsSync(rutaCatalog)) {
            // Para ES modules, necesitamos usar import() dinámico
            // En este contexto, simularemos la carga
            validacion_resultado.integracion.modelos_cargables++;
            validacion_resultado.integracion.pruebas_exitosas.push(`Modelo ${modelo} disponible en catalog/`);
            log_step('7.1', `✅ ${modelo} disponible en catalog/`);
        } else if (fs.existsSync(rutaMain)) {
            validacion_resultado.integracion.modelos_cargables++;
            validacion_resultado.integracion.pruebas_exitosas.push(`Modelo ${modelo} disponible en main/`);
            log_step('7.1', `✅ ${modelo} disponible en main/`);
        } else {
            validacion_resultado.integracion.errores_criticos.push(`Modelo ${modelo} no encontrado`);
            log_step('7.1', `❌ Modelo ${modelo} no encontrado`, 'error');
        }
    } catch (error) {
        validacion_resultado.integracion.errores_criticos.push(`Error cargando ${modelo}: ${error.message}`);
        log_step('7.1', `❌ Error con ${modelo}: ${error.message}`, 'error');
    }
}

// Prueba 2: Verificar servicios principales
log_step('7.2', 'Verificando servicios principales...');

const servicios_criticos = [
    'src/services/UsuarioService.js',
    'src/services/catalog/DepartamentoService.js',
    'src/services/catalog/MunicipioService.js',
    'src/services/ParroquiaService.js'
];

for (const servicio of servicios_criticos) {
    if (fs.existsSync(servicio)) {
        validacion_resultado.integracion.servicios_funcionales++;
        validacion_resultado.integracion.pruebas_exitosas.push(`Servicio ${path.basename(servicio)} disponible`);
        log_step('7.2', `✅ ${path.basename(servicio)} disponible`);
    } else {
        validacion_resultado.integracion.errores_criticos.push(`Servicio ${servicio} no encontrado`);
        log_step('7.2', `❌ ${servicio} no encontrado`, 'error');
    }
}

// Prueba 3: Verificar rutas principales
log_step('7.3', 'Verificando rutas principales...');

const rutas_criticas = [
    'src/routes/auth.js',
    'src/routes/usuarios.js',
    'src/routes/catalog.js',
    'src/routes/parroquias.js'
];

for (const ruta of rutas_criticas) {
    if (fs.existsSync(ruta)) {
        validacion_resultado.integracion.rutas_disponibles++;
        validacion_resultado.integracion.pruebas_exitosas.push(`Ruta ${path.basename(ruta)} disponible`);
        log_step('7.3', `✅ ${path.basename(ruta)} disponible`);
    } else {
        validacion_resultado.integracion.warnings.push(`Ruta ${ruta} no encontrada`);
        log_step('7.3', `⚠️ ${ruta} no encontrada`, 'warning');
    }
}

// Prueba 4: Verificar archivos de configuración
log_step('7.4', 'Verificando configuración...');

const configs_necesarios = [
    'src/config/database.js',
    'src/config/auth.js',
    '.env'
];

for (const config of configs_necesarios) {
    if (fs.existsSync(config)) {
        validacion_resultado.integracion.pruebas_exitosas.push(`Config ${path.basename(config)} disponible`);
        log_step('7.4', `✅ ${path.basename(config)} disponible`);
    } else {
        validacion_resultado.integracion.warnings.push(`Config ${config} no encontrado`);
        log_step('7.4', `⚠️ ${config} no encontrado`, 'warning');
    }
}

log_step('7.5', `Pruebas completadas: ${validacion_resultado.integracion.pruebas_exitosas.length} exitosas, ${validacion_resultado.integracion.errores_criticos.length} errores críticos`);

## 8. 📊 Validation Report
Generar reporte completo de validación con recomendaciones.

In [ ]:
// Generar reporte completo de validación
log_step('8.1', 'Generando reporte final de validación...');

// Calcular métricas generales
const total_errores = Object.values(validacion_resultado).reduce((sum, seccion) => {
    if (Array.isArray(seccion.errores)) return sum + seccion.errores.length;
    if (Array.isArray(seccion.errores_criticos)) return sum + seccion.errores_criticos.length;
    return sum;
}, 0);

const total_warnings = Object.values(validacion_resultado).reduce((sum, seccion) => {
    if (Array.isArray(seccion.warnings)) return sum + seccion.warnings.length;
    return sum;
}, 0);

// Determinar estado general
const estado_general = total_errores === 0 ? 'ESTABLE' : 
                      total_errores <= 3 ? 'ADVERTENCIA' : 'CRITICO';

const color_estado = estado_general === 'ESTABLE' ? 'green' : 
                    estado_general === 'ADVERTENCIA' ? 'orange' : 'red';

console.log('\\n' + '='.repeat(60));
console.log(`%c🎯 REPORTE DE VALIDACIÓN - ESTRUCTURA ACTUAL`, `color: ${color_estado}; font-weight: bold; font-size: 16px;`);
console.log('='.repeat(60));

console.log(`\\n📋 RESUMEN EJECUTIVO:`);
console.log(`   Estado General: %c${estado_general}`, `color: ${color_estado}; font-weight: bold;`);
console.log(`   Total Errores: ${total_errores}`);
console.log(`   Total Warnings: ${total_warnings}`);
console.log(`   Fecha: ${new Date().toLocaleString()}`);

// Reporte por secciones
console.log(`\\n📂 ESTRUCTURA DE MODELOS:`);
console.log(`   • Catalog: ${validacion_resultado.estructura.catalog.total_archivos} archivos`);
console.log(`   • Main: ${validacion_resultado.estructura.main.total_archivos} archivos`);
console.log(`   • Duplicados detectados: ${validacion_resultado.estructura.duplicados.length}`);
console.log(`   • Estado: ${validacion_resultado.estructura.catalog.total_archivos > 0 && validacion_resultado.estructura.main.total_archivos > 0 ? '✅ Dual structure OK' : '❌ Structure incomplete'}`);

console.log(`\\n🔌 CONEXIÓN BASE DE DATOS:`);
console.log(`   • Configuración: ${validacion_resultado.database.config_loaded ? '✅ Cargada' : '❌ Error'}`);
console.log(`   • Variables ENV: ${validacion_resultado.database.env_variables ? '✅ Disponibles' : '❌ Faltantes'}`);

if (validacion_resultado.servicios_rutas) {
    console.log(`\\n🛣️ SERVICIOS Y RUTAS:`);
    console.log(`   • Total servicios: ${validacion_resultado.servicios_rutas.total_servicios}`);
    console.log(`   • Imports problemáticos: ${validacion_resultado.servicios_rutas.rutas_consolidadas}`);
    console.log(`   • Estado: ${validacion_resultado.servicios_rutas.rutas_consolidadas === 0 ? '✅ Sin referencias a consolidated' : '⚠️ Necesita limpieza'}`);
}

if (validacion_resultado.dependencias) {
    console.log(`\\n📦 DEPENDENCIAS:`);
    console.log(`   • Node.js: ${validacion_resultado.dependencias.version_node}`);
    console.log(`   • Dependencias principales: ${validacion_resultado.dependencias.dependencias_principales.length}`);
    console.log(`   • Faltantes: ${validacion_resultado.dependencias.dependencias_faltantes.length}`);
    console.log(`   • Compatible: ${validacion_resultado.dependencias.compatible ? '✅ Sí' : '❌ No'}`);
}

if (validacion_resultado.integracion) {
    console.log(`\\n🔬 PRUEBAS DE INTEGRACIÓN:`);
    console.log(`   • Modelos cargables: ${validacion_resultado.integracion.modelos_cargables}`);
    console.log(`   • Servicios funcionales: ${validacion_resultado.integracion.servicios_funcionales}`);
    console.log(`   • Rutas disponibles: ${validacion_resultado.integracion.rutas_disponibles}`);
    console.log(`   • Errores críticos: ${validacion_resultado.integracion.errores_criticos.length}`);
}

// Recomendaciones
console.log(`\\n💡 RECOMENDACIONES:`);

if (estado_general === 'ESTABLE') {
    console.log(`   %c✅ La aplicación está en estado estable`, 'color: green;');
    console.log(`   %c✅ Estructura dual funcionando correctamente`, 'color: green;');
    console.log(`   %c✅ Listo para consolidación cuidadosa de modelos`, 'color: green;');
} else {
    console.log(`   %c⚠️ Requiere atención antes de consolidación`, 'color: orange;');
    
    if (total_errores > 0) {
        console.log(`   %c❌ Resolver ${total_errores} errores críticos primero`, 'color: red;');
    }
    
    if (validacion_resultado.servicios_rutas && validacion_resultado.servicios_rutas.rutas_consolidadas > 0) {
        console.log(`   %c🔧 Limpiar ${validacion_resultado.servicios_rutas.rutas_consolidadas} referencias a 'consolidated'`, 'color: orange;');
    }
    
    if (validacion_resultado.dependencias && !validacion_resultado.dependencias.compatible) {
        console.log(`   %c📦 Revisar dependencias del proyecto`, 'color: orange;');
    }
}

console.log(`\\n🚀 PRÓXIMOS PASOS:`);
if (estado_general === 'ESTABLE') {
    console.log(`   1. Backup del estado actual`);
    console.log(`   2. Planificar estrategia de consolidación`);
    console.log(`   3. Crear tests automatizados`);
    console.log(`   4. Ejecutar consolidación incremental`);
} else {
    console.log(`   1. Resolver errores críticos identificados`);
    console.log(`   2. Ejecutar este notebook nuevamente`);
    console.log(`   3. Verificar estado ESTABLE antes de consolidar`);
}

console.log('\\n' + '='.repeat(60));
console.log(`%c✨ Validación completada - ${new Date().toLocaleString()}`, 'color: blue; font-weight: bold;');
console.log('='.repeat(60));

// Guardar resultado en archivo
const reporte_completo = {
    fecha: new Date().toISOString(),
    estado_general,
    total_errores,
    total_warnings,
    validacion_resultado,
    recomendaciones: estado_general === 'ESTABLE' ? [
        'Aplicación estable y lista para consolidación',
        'Estructura dual funcionando correctamente',
        'Proceder con consolidación cuidadosa'
    ] : [
        'Resolver errores críticos antes de consolidar',
        'Limpiar referencias problemáticas',
        'Verificar dependencias del proyecto'
    ]
};

try {
    fs.writeFileSync('validacion-resultado.json', JSON.stringify(reporte_completo, null, 2));
    log_step('8.2', '✅ Reporte guardado en validacion-resultado.json');
} catch (error) {
    log_step('8.2', `⚠️ No se pudo guardar el reporte: ${error.message}`, 'warning');
}

console.log(`\\n📄 Resultado final disponible en: validacion-resultado.json`);